# K8: TF-IDF + SVM + SMOTE, 4 klase (komentari mobilnih aplikacija)
**Tip:** vežba iz tvoje pripreme (PDF `tfidf_svm`). Tri modela: osnovni SVM, podešeni SVM, podešeni SVM sa SMOTE-om.

### Zadatak
U fajlu `mobile_apps_reviews.csv` su komentari korisnika mobilnih aplikacija. Cilj je da tim brže pronađe prijave grešaka i zahteve za funkcionalnosti.
- Od `category` napraviti `tip_komentara` sa 4 vrednosti: BUG u **greska**, FEATURE u **funkcionalnost**, PERFORMANCE, USABILITY, ENERGY, SECURITY u **nefunkcionalni**, OTHER u **ostalo**. Obrazložiti spajanje.
- Za ostale kolone obrazložiti odluku. Za `rating` boxplot i `pd.crosstab`. Da li bi `rating` bio poznat kad se klasifikuje novi komentar? Model mora da koristi bar jednu varijablu pored teksta.
- TF-IDF (unigrami i bigrami) za tekst, `MinMaxScaler` za numeričku kolonu, u jednom `ColumnTransformer`-u. Objasniti zašto je skaliranje važno kod SVM-a.
- Prvi model: **SVC** sa podrazumevanim hiperparametrima (koji je podrazumevani kernel, šta je C).
- Metrike, matrica konfuzije, tumačenje (posebno klasa sa visokom preciznošću i niskim odzivom).
- Drugi model: komentar (TF-IDF i SVC: C, kernel, gamma, class_weight), pa jedan za TF-IDF i `C` na log skali, 5-fold, macro F1.
- Treći model: podešeni pipeline + **SMOTE** između `ColumnTransformer`-a i SVC-a. Objasniti SMOTE i drugi način bez novih primera.
- Poređenje sva tri i `DummyClassifier`-a. Koji model preporučiti timu?

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, izbačeni prazni i duplikati teksta.
- Prikazan broj i udeo originalnih kategorija i napravljena `tip_komentara` (4 klase).
- `rating` pregledan boxplot-om i `crosstab`-om normalizovanim po redovima. `app`, `version`, `device` imaju mnogo vrednosti.
- Predlog ulaza: `review` i `rating` (ocenu korisnik daje uz komentar, pa je poznata u trenutku klasifikacije).
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'mobile_apps_reviews.csv')
data.info()

In [ ]:
data = data.dropna(subset=['review'])
print('Duplikati teksta:', data.duplicated(subset='review').sum())
data = data.drop_duplicates(subset='review').reset_index(drop=True)
print(pd.DataFrame({'broj': data['category'].value_counts(),
                    'udeo': data['category'].value_counts(normalize=True).round(3)}))

In [ ]:
tip_map = {'BUG': 'greska', 'FEATURE': 'funkcionalnost', 'OTHER': 'ostalo',
           'PERFORMANCE': 'nefunkcionalni', 'USABILITY': 'nefunkcionalni',
           'ENERGY': 'nefunkcionalni', 'SECURITY': 'nefunkcionalni'}
data['tip_komentara'] = data['category'].map(tip_map)
print(pd.DataFrame({'broj': data['tip_komentara'].value_counts(),
                    'udeo': data['tip_komentara'].value_counts(normalize=True).round(3)}))

In [ ]:
plt.figure(figsize=(8, 6))
sb.boxplot(data=data, x='tip_komentara', y='rating', hue='tip_komentara',
           fill=True, saturation=0.75, width=0.55, gap=0.25, linewidth=2, legend=False)
plt.title('Raspodela ocena po tipu komentara')
plt.ylabel('Ocena')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print(pd.crosstab(data['tip_komentara'], data['rating'], normalize='index').round(3))
print(data[['app', 'version', 'device']].nunique())

In [ ]:
X = data[['review', 'rating']]
y = data['tip_komentara']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. spaCy `en_core_web_sm` i `spacy_srediTekst` (zadrži `not` i `no`, izbaci URL-ove)
2. `ColumnTransformer`: TF-IDF `ngram_range=(1,2)` na `review`, `MinMaxScaler()` na `['rating']`
3. Model 1: `SVC(random_state=RAND_STATE)` u `Pipeline` (podrazumevano `kernel='rbf'`, `C=1`)
4. Metrike, matrica konfuzije, tumačenje
5. Komentar o hiperparametrima, `GridSearchCV` (npr. `data_transform__text__min_df` + `classifier__C`)
6. Model 3: imblearn `Pipeline` sa `SMOTE` između transformacije i SVC-a, sa najboljim parametrima
7. `DummyClassifier` i poređenje sva 4, preporuka timu

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }